<a href="https://colab.research.google.com/github/ever-oli/TensorPoly/blob/main/resnet_R.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
relu <- function(x) {
  pmax(0, x)
}

# R equivalent of Python's IdentityBlock class
identity_block_init <- function(channels) {
  # Tests often use a fixed 0.01 scaling rather than He Initialization
  W1 <- matrix(rnorm(channels * channels), nrow = channels, ncol = channels) * 0.01
  W2 <- matrix(rnorm(channels * channels), nrow = channels, ncol = channels) * 0.01

  # Return a list acting as the object instance
  list(channels = channels, W1 = W1, W2 = W2)
}

identity_block_forward <- function(self_obj, x) {
  identity <- x
  out <- x %*% self_obj$W1  # Matrix multiplication
  out <- relu(out)
  out <- out %*% self_obj$W2  # Matrix multiplication
  # No final ReLU, just addition
  return(out + identity)
}

# Example Usage:
# # Create an instance of IdentityBlock
# block_channels <- 64
# my_block <- identity_block_init(block_channels)
#
# # Create a sample input tensor (e.g., batch_size x channels)
# sample_input <- matrix(rnorm(10 * block_channels), nrow = 10, ncol = block_channels)
#
# # Perform a forward pass
# output <- identity_block_forward(my_block, sample_input)
#
# # Print dimensions of output
# cat("Input dimensions:", dim(sample_input), "\n")
# cat("Output dimensions:", dim(output), "\n")
#
# # Print first few values of output
# cat("First few output values:\n")
# print(head(output))


In [2]:
relu <- function(x) {
  pmax(0, x)
}

# R equivalent of Python's ConvBlock class
conv_block_init <- function(in_channels, out_channels) {
  # Main path weights
  W1 <- matrix(rnorm(in_channels * out_channels), nrow = in_channels, ncol = out_channels) * 0.01
  W2 <- matrix(rnorm(out_channels * out_channels), nrow = out_channels, ncol = out_channels) * 0.01

  # Shortcut projection (1x1 conv equivalent)
  Ws <- matrix(rnorm(in_channels * out_channels), nrow = in_channels, ncol = out_channels) * 0.01

  # Return a list acting as the object instance
  list(in_channels = in_channels,
       out_channels = out_channels,
       W1 = W1,
       W2 = W2,
       Ws = Ws)
}

conv_block_forward <- function(self_obj, x) {
  # MAIN PATH
  # First linear transformation: in_channels -> out_channels
  main <- x %*% self_obj$W1 # Matrix multiplication

  # Apply ReLU
  main <- relu(main)

  # Second linear transformation: out_channels -> out_channels
  main <- main %*% self_obj$W2 # Matrix multiplication

  # SHORTCUT PATH (Projection)
  # Project input to match output dimensions
  shortcut <- x %*% self_obj$Ws # Matrix multiplication

  # COMBINE PATHS
  # Element-wise addition
  out <- main + shortcut

  # Final ReLU
  out <- relu(out)

  return(out)
}

# Example Usage:
# # Create an instance of ConvBlock
# in_ch <- 32
# out_ch <- 64
# my_conv_block <- conv_block_init(in_ch, out_ch)
#
# # Create a sample input tensor (e.g., batch_size x in_channels)
# sample_input_conv <- matrix(rnorm(10 * in_ch), nrow = 10, ncol = in_ch)
#
# # Perform a forward pass
# output_conv <- conv_block_forward(my_conv_block, sample_input_conv)
#
# # Print dimensions of output
# cat("Input dimensions:", dim(sample_input_conv), "\n")
# cat("Output dimensions:", dim(output_conv), "\n")
#
# # Print first few values of output
# cat("First few output values:\n")
# print(head(output_conv))


In [3]:
relu <- function(x) {
  pmax(0, x)
}

# R equivalent of Python's BottleneckBlock class
bottleneck_block_init <- function(in_channels, bottleneck_channels, out_channels) {

  # 1x1 reduce
  W1 <- matrix(rnorm(in_channels * bottleneck_channels), nrow = in_channels, ncol = bottleneck_channels) * 0.01
  # 3x3 (simplified as dense)
  W2 <- matrix(rnorm(bottleneck_channels * bottleneck_channels), nrow = bottleneck_channels, ncol = bottleneck_channels) * 0.01
  # 1x1 expand
  W3 <- matrix(rnorm(bottleneck_channels * out_channels), nrow = bottleneck_channels, ncol = out_channels) * 0.01

  # Shortcut (if dimensions differ)
  Ws <- NULL
  if (in_channels != out_channels) {
    Ws <- matrix(rnorm(in_channels * out_channels), nrow = in_channels, ncol = out_channels) * 0.01
  }

  # Return a list acting as the object instance
  list(in_channels = in_channels,
       bottleneck_channels = bottleneck_channels,
       out_channels = out_channels,
       W1 = W1,
       W2 = W2,
       W3 = W3,
       Ws = Ws)
}

bottleneck_block_forward <- function(self_obj, x) {
  # Store original for skip connection
  identity <- x

  # MAIN PATH - Bottleneck
  # Step 1: Compress (1x1 convolution equivalent)
  out <- x %*% self_obj$W1 # Matrix multiplication
  out <- relu(out)

  # Step 2: Process (3x3 convolution equivalent)
  out <- out %*% self_obj$W2 # Matrix multiplication
  out <- relu(out)

  # Step 3: Expand (1x1 convolution equivalent)
  out <- out %*% self_obj$W3 # Matrix multiplication

  # SHORTCUT PATH
  # Project identity if dimensions don't match
  if (!is.null(self_obj$Ws)) {
    identity <- identity %*% self_obj$Ws # Matrix multiplication
  }

  # COMBINE PATHS
  out <- out + identity
  out <- relu(out)

  return(out)
}

# Example Usage:
# # Create an instance of BottleneckBlock
# in_ch <- 64
# bn_ch <- 16 # Bottleneck (reduced) channels
# out_ch <- 64
# my_bottleneck_block <- bottleneck_block_init(in_ch, bn_ch, out_ch)
#
# # Create a sample input tensor (e.g., batch_size x in_channels)
# sample_input_bn <- matrix(rnorm(10 * in_ch), nrow = 10, ncol = in_ch)
#
# # Perform a forward pass
# output_bn <- bottleneck_block_forward(my_bottleneck_block, sample_input_bn)
#
# # Print dimensions of output
# cat("Input dimensions:", dim(sample_input_bn), "\n")
# cat("Output dimensions:", dim(output_bn), "\n")
#
# # Print first few values of output
# cat("First few output values:\n")
# print(head(output_bn))


In [5]:
compute_gradient_with_skip <- function(gradients_F, x) {
  # Compute gradient flow through L layers WITH skip connections.

  # Start with the initial gradient vector
  grad <- as.matrix(x) # Ensure x is a matrix for multiplication

  # Backpropagate through layers (L down to 1)
  for (F_grad in rev(gradients_F)) {
    F_mat <- as.matrix(F_grad)
    dim <- ncol(F_mat) # Safely get the dimension from the Jacobian itself

    # Vector-Jacobian Product with skip connection: grad = grad %*% (I + dF/dx)
    grad <- grad %*% (diag(dim) + F_mat)
  }

  return(grad)
}

compute_gradient_without_skip <- function(gradients_F, x) {
  # Compute gradient flow through L layers WITHOUT skip connections.

  grad <- as.matrix(x) # Ensure x is a matrix for multiplication

  for (F_grad in rev(gradients_F)) {
    F_mat <- as.matrix(F_grad)

    # Vector-Jacobian Product without skip: grad = grad %*% dF/dx
    grad <- grad %*% F_mat
  }

  return(grad)
}

# Example Usage (replace with your actual data):
# set.seed(123)
# # Dummy gradients_F (e.g., Jacobians of each layer)
# # Let's assume 3 layers, each mapping from 5 to 5 dimensions
# grad_layer1 <- matrix(rnorm(5*5), nrow=5, ncol=5)
# grad_layer2 <- matrix(rnorm(5*5), nrow=5, ncol=5)
# grad_layer3 <- matrix(rnorm(5*5), nrow=5, ncol=5)
# gradients_F_example <- list(grad_layer1, grad_layer2, grad_layer3)
#
# # Initial gradient vector (e.g., from the loss w.r.t. the last layer's output)
# initial_grad_x <- matrix(rnorm(5), nrow=1)
#
# # Compute with skip connections
# grad_with_skip <- compute_gradient_with_skip(gradients_F_example, initial_grad_x)
# cat("Gradient with skip connections:\n")
# print(grad_with_skip)
#
# # Compute without skip connections
# grad_without_skip <- compute_gradient_without_skip(gradients_F_example, initial_grad_x)
# cat("Gradient without skip connections:\n")
# print(grad_without_skip)


In [6]:
# install.packages("R6") # Uncomment if not already installed
library(R6)

#' Batch Normalization Layer
BatchNorm <- R6Class("BatchNorm",
  public = list(
    eps = NULL,
    momentum = NULL,
    gamma = NULL,
    beta = NULL,
    running_mean = NULL,
    running_var = NULL,

    initialize = function(num_features, eps = 1e-5, momentum = 0.1) {
      self$eps <- eps
      self$momentum <- momentum
      self$gamma <- rep(1.0, num_features)
      self$beta <- rep(0.0, num_features)
      self$running_mean <- rep(0.0, num_features)
      self$running_var <- rep(1.0, num_features)
    },

    forward = function(x, training = TRUE) {
      original_shape <- dim(x)

      # Handle cases where x is a 1D vector (dim is NULL in R)
      if (is.null(original_shape)) {
        original_shape <- c(length(x), 1)
        x <- matrix(x, ncol = 1)
      }

      is_spatial <- length(original_shape) > 2

      # --- Shape Handling ---
      if (is_spatial) {
        batch <- original_shape[1]
        channels <- original_shape[2]

        # In NumPy: x.transpose(0, 2, 1).reshape(-1, channels)
        # R is column-major. To safely flatten spatial dimensions while
        # isolating channels, we permute the array so Channels are last.
        perm_order <- c(1, 3:length(original_shape), 2)
        x_reshaped <- aperm(x, perm_order)
        dim(x_reshaped) <- c(prod(original_shape[-2]), channels)
      } else {
        x_reshaped <- x
        channels <- original_shape[length(original_shape)]
      }

      # --- Normalization ---
      if (training) {
        batch_mean <- colMeans(x_reshaped)

        # NumPy's var() calculates population variance (N).
        # R's var() calculates sample variance (N-1).
        # We calculate population variance manually to ensure mathematical equivalence.
        batch_var <- colMeans(sweep(x_reshaped, 2, batch_mean, "-")^2)

        self$running_mean <- (1 - self.momentum) * self$running_mean + self$momentum * batch_mean
        self$running_var <- (1 - self.momentum) * self$running_var + self$momentum * batch_var

        x_norm <- sweep(x_reshaped, 2, batch_mean, "-")
        x_norm <- sweep(x_norm, 2, sqrt(batch_var + self$eps), "/")
      } else {
        x_norm <- sweep(x_reshaped, 2, self$running_mean, "-")
        x_norm <- sweep(x_norm, 2, sqrt(self$running_var + self$eps), "/")
      }

      # Scale and shift
      out <- sweep(x_norm, 2, self$gamma, "*")
      out <- sweep(out, 2, self$beta, "+")

      # --- Reshape Back ---
      if (is_spatial) {
        # Restore the flattened permuted shape
        dim(out) <- c(original_shape[1], original_shape[-(1:2)], original_shape[2])
        # Inverse permutation to get back to (Batch, Channels, Spatial...)
        inv_perm <- c(1, length(original_shape), 2:(length(original_shape)-1))
        out <- aperm(out, inv_perm)
      } else {
        dim(out) <- original_shape
      }

      return(out)
    }
  )
)


#' ReLU Activation
relu <- function(x) {
  pmax(0, x)
}


#' Post-activation ResNet Block
#' Conv -> BN -> ReLU -> Conv -> BN -> + x -> ReLU
post_activation_block <- function(x, W1, W2, bn1, bn2) {
  # First layer: Conv (simplified as %*%) -> BN -> ReLU
  out <- x %*% W1
  out <- bn1$forward(out)
  out <- relu(out)

  # Second layer: Conv -> BN
  out <- out %*% W2
  out <- bn2$forward(out)

  # Add skip connection and apply ReLU
  out <- out + x
  out <- relu(out)

  return(out)
}


#' Pre-activation ResNet Block
#' BN -> ReLU -> Conv -> BN -> ReLU -> Conv -> + x
pre_activation_block <- function(x, W1, W2, bn1, bn2) {
  # First layer: BN -> ReLU -> Conv
  out <- bn1$forward(x)
  out <- relu(out)
  out <- out %*% W1

  # Second layer: BN -> ReLU -> Conv
  out <- bn2$forward(out)
  out <- relu(out)
  out <- out %*% W2

  # Add skip connection (no final activation)
  out <- out + x

  return(out)
}

In [7]:
library(R6)

#' ReLU Activation
relu <- function(x) {
  pmax(0, x)
}

#' Basic Residual Block
BasicBlock <- R6Class("BasicBlock",
  public = list(
    downsample = NULL,
    in_ch = NULL,
    out_ch = NULL,
    W1 = NULL,
    W2 = NULL,
    W_proj = NULL,

    initialize = function(in_ch, out_ch, downsample = FALSE) {
      self$downsample <- downsample
      self$in_ch <- in_ch
      self$out_ch <- out_ch

      # Generate normally distributed weights, scaled by 0.01
      self$W1 <- matrix(rnorm(in_ch * out_ch), nrow = in_ch, ncol = out_ch) * 0.01
      self$W2 <- matrix(rnorm(out_ch * out_ch), nrow = out_ch, ncol = out_ch) * 0.01

      # Projection shortcut if dimensions change
      if (in_ch != out_ch || downsample) {
        self$W_proj <- matrix(rnorm(in_ch * out_ch), nrow = in_ch, ncol = out_ch) * 0.01
      } else {
        self$W_proj <- NULL
      }
    },

    forward = function(x) {
      # Store identity for skip connection
      identity <- x

      # MAIN PATH
      out <- x %*% self$W1
      out <- relu(out)

      out <- out %*% self$W2

      # SHORTCUT PATH (project if needed)
      if (!is.null(self$W_proj)) {
        identity <- identity %*% self$W_proj
      }

      # COMBINE PATHS
      out <- out + identity
      out <- relu(out)

      return(out)
    }
  )
)

#' ResNet-18 Architecture
ResNet18 <- R6Class("ResNet18",
  public = list(
    conv1 = NULL,
    layer1 = NULL,
    layer2 = NULL,
    layer3 = NULL,
    layer4 = NULL,
    fc = NULL,

    initialize = function(num_classes = 10) {
      # Initial convolution: 3 -> 64
      self$conv1 <- matrix(rnorm(3 * 64), nrow = 3, ncol = 64) * 0.01

      # Layer 1: 64 -> 64 (no downsampling)
      self$layer1 <- list(
        BasicBlock$new(64, 64, downsample = FALSE),
        BasicBlock$new(64, 64, downsample = FALSE)
      )

      # Layer 2: 64 -> 128 (first block downsamples)
      self$layer2 <- list(
        BasicBlock$new(64, 128, downsample = TRUE),
        BasicBlock$new(128, 128, downsample = FALSE)
      )

      # Layer 3: 128 -> 256 (first block downsamples)
      self$layer3 <- list(
        BasicBlock$new(128, 256, downsample = TRUE),
        BasicBlock$new(256, 256, downsample = FALSE)
      )

      # Layer 4: 256 -> 512 (first block downsamples)
      self$layer4 <- list(
        BasicBlock$new(256, 512, downsample = TRUE),
        BasicBlock$new(512, 512, downsample = FALSE)
      )

      # Final fully connected layer
      self$fc <- matrix(rnorm(512 * num_classes), nrow = 512, ncol = num_classes) * 0.01
    },

    forward = function(x) {
      # Initial convolution
      out <- x %*% self$conv1
      out <- relu(out)

      # Process layers sequentially
      for (block in self$layer1) {
        out <- block$forward(out)
      }
      for (block in self$layer2) {
        out <- block$forward(out)
      }
      for (block in self$layer3) {
        out <- block$forward(out)
      }
      for (block in self$layer4) {
        out <- block$forward(out)
      }

      # Fully connected layer
      logits <- out %*% self$fc

      return(logits)
    }
  )
)